# SIMD Membase编程基础与常见指令

在前面的[Ascend C 算子开发系列教程](../ascendc_operator_development/02_AscendC_basic/README.md)中已经学习了Ascend C的编程范式和核函数的基本概念。本节将深入探讨Ascend C算子开发中最核心的编程模式——**SIMD Membase（基于内存的编程范式）**。

Membase是绝大多数算子开发的起点与基石。无论是刚入门的初学者还是追求高效交付的资深开发者，Membase范式都以其封装完善、开发便捷的优势，成为日常算子开发的首选方案。本节将通过理论讲解和实际代码运行，系统性地拆解Membase编程的核心原理，从硬件内存架构到编程范式，从关键概念（DataBlock、Repeat、Mask）到常见指令的使用方法。

本节学习大纲如下：
- SIMD与Membase的基本概念
- SIMD Vector内存架构与数据流
- Membase编程范式详解（含完整代码案例）
- 核心概念深度解析：DataBlock、Repeat、Mask
- 常见SIMD Membase指令实战演练
- Membase编程最佳实践
---
## 1. 环境准备

正式开始学习之前，先要对jupyter环境进行初始化。以下代码完成了初始化并将环境中的变量导入jupyter环境，同时完成了代码目录的创建。

In [ ]:
!mkdir -p Sources/02.06

import os, subprocess
env = subprocess.check_output("bash -l -c 'source $ASCEND_TOOLKIT_HOME/set_env.sh && env'", shell=True, text=True)
for line in env.splitlines():
    if "=" in line: os.environ.__setitem__(*line.split("=", 1))

---

## 2. 什么是SIMD Membase？

### 2.1 SIMD的定义

**SIMD**（Single Instruction, Multiple Data，单指令多数据）是一种并行计算模式：一条指令同时对多个数据执行相同操作。昇腾AI Core的Vector计算单元正是基于SIMD架构设计的。

举个例子：当执行`Add(zLocal, xLocal, yLocal, count)`时，Vector计算单元会同时对`count`个元素执行加法操作，而不是逐个元素循环计算。这就是SIMD的威力！

### 2.2 Membase的定义

**Membase**（基于内存）是Ascend C的两种核心编程范式之一：

| 编程范式 | 数据载体 | 控制方式 | 适用场景 |
|---------|---------|---------|---------|
| **Membase（基于内存）** | LocalTensor | API封装搬运与计算，开发者关注数据流 | 日常开发、快速交付 |
| **Regbase（基于寄存器）** | Reg矢量计算寄存器 | 手动控制Load/Store/Compute，精细化操作 | 追求极致性能、特殊计算模式 |

Membase的核心特点是：**所有操作均基于内存进行**，每次计算都需要从Local Memory（主要是Unified Buffer）加载数据，计算完成后将结果搬回Local Memory，中间计算结果暂存在Local Memory上。API自动处理这些数据搬运过程，开发者无需关心底层寄存器的操作细节。

### 2.3 Membase调用层级

在Membase架构中，API的调用层级如下：

```
高阶API → 基础API → 框架API → 编译器BuiltIn API
```

- **高阶API**：如Matmul，封装完整的矩阵计算流程
- **基础API**：如Add、Mul、DataCopy等，输入输出均为LocalTensor
- **框架API**：编译器层面的封装，最终调用BuiltIn API

开发者主要使用的是**基础API**，这也是本节的重点内容。

---

## 3. SIMD Vector内存架构

### 3.1 内存层级结构

理解内存架构是Membase编程的基础。数据在不同层级间流动，构成了完整的计算流程：

```
┌────────────────────────────────────────────────┐
│               Global Memory (GM)               │
│       Large capacity (GBs), slow access        │
│       Input data & final output storage        │
└──────────────────┬─────────────────────────────┘
                   │ DataCopy (MTE2/MTE3)
                   ↓
┌────────────────────────────────────────────────┐
│              Unified Buffer (UB)               │
│         Per-core shared, medium speed          │
│     Vector compute input/output/temp data      │
│       Maps to: VECIN / VECOUT / VECCALC        │
└──────────────────┬─────────────────────────────┘
                   │ Auto hardware transfer
                   ↓
┌────────────────────────────────────────────────┐
│              Vector Compute Unit               │
│            SIMD parallel execution             │
│    One instruction processes multiple data     │
└────────────────────────────────────────────────┘
```

### 3.2 关键内存概念：TPosition

| 逻辑位置（TPosition） | 物理存储 | 用途 |
|---|---|---|
| VECIN | Unified Buffer | 矢量计算输入数据 |
| VECCALC | Unified Buffer | 矢量计算临时变量 |
| VECOUT | Unified Buffer | 矢量计算输出数据 |

**核心要点**：VECIN、VECCALC、VECOUT虽然在逻辑上是不同位置，但物理上都对应同一块Unified Buffer内存。Ascend C通过TPosition抽象来区分它们的使用角色。

### 3.3 数据流总览

Membase编程中，典型的数据流动路径：

```
GM → (DataCopy MTE2) → UB(VECIN) → (Vector计算) → UB(VECOUT) → (DataCopy MTE3) → GM
```

---

## 4. Membase编程范式详解与实战

### 4.1 流水线编程范式

Ascend C的Membase编程采用流水线式编程范式，将算子核内的处理程序分为多个**流水任务**，通过队列（TQue）完成任务间通信和同步，并通过TPipe统一管理内存等资源。

矢量编程范式将算子实现分为3个基本任务：

```
┌──────────┐    ┌──────────┐    ┌──────────┐
│  CopyIn  │ →  │ Compute  │ →  │ CopyOut  │
│ GM → UB  │    │ vector   │    │ UB → GM  │
└──────────┘    └──────────┘    └──────────┘
```

### 4.2 实战案例：向量加法算子（完整可运行代码）

下面我们通过一个完整的向量加法算子来演示Membase编程的完整流程。这是最经典的Membase编程案例，涵盖了CopyIn、Compute、CopyOut三段式流水线的所有核心要素。

In [ ]:
%%writefile Sources/02.06/add_custom.asc

#include <cstdint>
#include <iostream>
#include <vector>
#include <algorithm>
#include <iterator>
#include "acl/acl.h"
#include "kernel_operator.h"

constexpr uint32_t BUFFER_NUM = 2;

struct AddCustomTilingData
{
    uint32_t totalLength;
    uint32_t tileNum;
};

class KernelAdd {
public:
    __aicore__ inline KernelAdd(){}
    __aicore__ inline void Init(GM_ADDR x, GM_ADDR y, GM_ADDR z, uint32_t totalLength, uint32_t tileNum);
    __aicore__ inline void Process();

private:
    __aicore__ inline void CopyIn(int32_t progress);
    __aicore__ inline void Compute(int32_t progress);
    __aicore__ inline void CopyOut(int32_t progress);

private:
    AscendC::TPipe pipe;
    AscendC::TQue<AscendC::TPosition::VECIN, BUFFER_NUM> inQueueX, inQueueY;
    AscendC::TQue<AscendC::TPosition::VECOUT, BUFFER_NUM> outQueueZ;
    AscendC::GlobalTensor<float> xGm;
    AscendC::GlobalTensor<float> yGm;
    AscendC::GlobalTensor<float> zGm;
    uint32_t blockLength;
    uint32_t tileNum;
    uint32_t tileLength;
};

**Init函数**：完成多核数据切分与内存分配

Init函数是Membase编程的起点，负责：
1. 多核并行切分：通过GetBlockIdx()计算每个核的数据偏移
2. 单核内数据切块：将数据切分为tileNum块，开启Double Buffer
3. 设置Global Memory起始地址
4. 为TQue队列分配UB内存

In [ ]:
%%writefile -a Sources/02.06/add_custom.asc

__aicore__ inline void KernelAdd::Init(GM_ADDR x, GM_ADDR y, GM_ADDR z, uint32_t totalLength, uint32_t tileNum)
{
    this->blockLength = totalLength / AscendC::GetBlockNum();
    this->tileNum = tileNum;
    this->tileLength = this->blockLength / tileNum / BUFFER_NUM;
    xGm.SetGlobalBuffer((__gm__ float *)x + this->blockLength * AscendC::GetBlockIdx(), this->blockLength);
    yGm.SetGlobalBuffer((__gm__ float *)y + this->blockLength * AscendC::GetBlockIdx(), this->blockLength);
    zGm.SetGlobalBuffer((__gm__ float *)z + this->blockLength * AscendC::GetBlockIdx(), this->blockLength);
    pipe.InitBuffer(inQueueX, BUFFER_NUM, this->tileLength * sizeof(float));
    pipe.InitBuffer(inQueueY, BUFFER_NUM, this->tileLength * sizeof(float));
    pipe.InitBuffer(outQueueZ, BUFFER_NUM, this->tileLength * sizeof(float));
}

**Process + CopyIn + Compute + CopyOut函数**：实现三级流水线

这是Membase编程的核心——三级流水线：
- CopyIn：从GM搬运数据到UB（VECIN），使用DataCopy
- Compute：从队列取出数据，执行矢量Add计算，放入输出队列
- CopyOut：从输出队列取出结果，搬运回GM

In [ ]:
%%writefile -a Sources/02.06/add_custom.asc

__aicore__ inline void KernelAdd::Process()
{
    int32_t loopCount = this->tileNum * BUFFER_NUM;
    for (int32_t i = 0; i < loopCount; i++) {
        CopyIn(i);
        Compute(i);
        CopyOut(i);
    }
}

__aicore__ inline void KernelAdd::CopyIn(int32_t progress)
{
    AscendC::LocalTensor<float> xLocal = inQueueX.AllocTensor<float>();
    AscendC::LocalTensor<float> yLocal = inQueueY.AllocTensor<float>();
    AscendC::DataCopy(xLocal, xGm[progress * this->tileLength], this->tileLength);
    AscendC::DataCopy(yLocal, yGm[progress * this->tileLength], this->tileLength);
    inQueueX.EnQue(xLocal);
    inQueueY.EnQue(yLocal);
}

__aicore__ inline void KernelAdd::Compute(int32_t progress)
{
    AscendC::LocalTensor<float> xLocal = inQueueX.DeQue<float>();
    AscendC::LocalTensor<float> yLocal = inQueueY.DeQue<float>();
    AscendC::LocalTensor<float> zLocal = outQueueZ.AllocTensor<float>();
    AscendC::Add(zLocal, xLocal, yLocal, this->tileLength);
    outQueueZ.EnQue<float>(zLocal);
    inQueueX.FreeTensor(xLocal);
    inQueueY.FreeTensor(yLocal);
}

__aicore__ inline void KernelAdd::CopyOut(int32_t progress)
{
    AscendC::LocalTensor<float> zLocal = outQueueZ.DeQue<float>();
    AscendC::DataCopy(zGm[progress * this->tileLength], zLocal, this->tileLength);
    outQueueZ.FreeTensor(zLocal);
}

__global__ __aicore__ void add_custom(GM_ADDR x, GM_ADDR y, GM_ADDR z, AddCustomTilingData tiling) {
    KERNEL_TASK_TYPE_DEFAULT(KERNEL_TYPE_AIV_ONLY);
    KernelAdd op;
    op.Init(x, y, z, tiling.totalLength, tiling.tileNum);
    op.Process();
}

**Host侧调用代码**：kernel_add函数与main函数

这部分代码在CPU上运行，负责初始化ACL环境、分配内存、调用核函数并验证结果。

In [ ]:
%%writefile -a Sources/02.06/add_custom.asc

std::vector<float> kernel_add(std::vector<float> &x, std::vector<float> &y)
{
    constexpr uint32_t blockDim = 8;
    uint32_t totalLength = x.size();
    size_t totalByteSize = totalLength * sizeof(float);
    int32_t deviceId = 0;
    aclrtStream stream = nullptr;
    AddCustomTilingData tiling = {totalLength, 8};
    uint8_t *xHost = reinterpret_cast<uint8_t *>(x.data());
    uint8_t *yHost = reinterpret_cast<uint8_t *>(y.data());
    uint8_t *zHost = nullptr;
    uint8_t *xDevice = nullptr;
    uint8_t *yDevice = nullptr;
    uint8_t *zDevice = nullptr;

    aclInit(nullptr);
    aclrtSetDevice(deviceId);
    aclrtCreateStream(&stream);

    aclrtMallocHost((void **)(&zHost), totalByteSize);
    aclrtMalloc((void **)&xDevice, totalByteSize, ACL_MEM_MALLOC_HUGE_FIRST);
    aclrtMalloc((void **)&yDevice, totalByteSize, ACL_MEM_MALLOC_HUGE_FIRST);
    aclrtMalloc((void **)&zDevice, totalByteSize, ACL_MEM_MALLOC_HUGE_FIRST);

    aclrtMemcpy(xDevice, totalByteSize, xHost, totalByteSize, ACL_MEMCPY_HOST_TO_DEVICE);
    aclrtMemcpy(yDevice, totalByteSize, yHost, totalByteSize, ACL_MEMCPY_HOST_TO_DEVICE);

    add_custom<<<blockDim, nullptr, stream>>>(xDevice, yDevice, zDevice, tiling);
    aclrtSynchronizeStream(stream);

    aclrtMemcpy(zHost, totalByteSize, zDevice, totalByteSize, ACL_MEMCPY_DEVICE_TO_HOST);
    std::vector<float> z((float *)zHost, (float *)(zHost + totalByteSize));

    aclrtFree(xDevice);
    aclrtFree(yDevice);
    aclrtFree(zDevice);
    aclrtFreeHost(zHost);
    aclrtDestroyStream(stream);
    aclrtResetDevice(deviceId);
    aclFinalize();
    return z;
}

uint32_t VerifyResult(std::vector<float> &output, std::vector<float> &golden)
{
    constexpr size_t maxPrintSize = 20;
    std::cout << "Output: ";
    std::copy(output.begin(), output.begin() + std::min(output.size(), maxPrintSize),
        std::ostream_iterator<float>(std::cout, " "));
    if (output.size() > maxPrintSize) std::cout << "...";
    std::cout << std::endl;
    std::cout << "Golden: ";
    std::copy(golden.begin(), golden.begin() + std::min(golden.size(), maxPrintSize),
        std::ostream_iterator<float>(std::cout, " "));
    if (golden.size() > maxPrintSize) std::cout << "...";
    std::cout << std::endl;
    if (std::equal(golden.begin(), golden.end(), output.begin())) {
        std::cout << "[Success] Add verification passed." << std::endl;
        return 0;
    } else {
        std::cout << "[Failed] Add verification failed!" << std::endl;
        return 1;
    }
}

int32_t main(int32_t argc, char *argv[])
{
    constexpr uint32_t totalLength = 8 * 2048;
    constexpr float valueX = 1.2f;
    constexpr float valueY = 2.3f;
    std::vector<float> x(totalLength, valueX);
    std::vector<float> y(totalLength, valueY);

    std::vector<float> output = kernel_add(x, y);

    std::vector<float> golden(totalLength, valueX + valueY);
    return VerifyResult(output, golden);
}

**编译并运行Add算子**

现在我们来编译和运行这个Membase编程范式下的Add算子，验证结果是否正确。

In [ ]:
%%writefile Sources/02.06/CMakeLists.txt

cmake_minimum_required(VERSION 3.16)
find_package(ASC REQUIRED)
project(kernel_samples LANGUAGES ASC CXX)

add_executable(add_demo
    add_custom.asc
)
target_compile_options(add_demo PRIVATE
    $<$<COMPILE_LANGUAGE:ASC>:--npu-arch=dav-2201>
)

In [ ]:
!cd Sources/02.06 && mkdir -p build
!export ASC_DIR=$ASCEND_HOME_PATH/aarch64-linux/tikcpp/ascendc_kernel_cmake/ && cd Sources/02.06/build/ && cmake .. && make

In [ ]:
!cd Sources/02.06/build/ && ./add_demo

---

## 5. 核心概念深度解析

理解以下概念是正确使用矢量计算API的基础。

### 5.1 DataBlock

**DataBlock**是矢量计算指令处理的数据单元，大小通常为**32字节**。

| 数据类型 | 每个DataBlock包含的元素数 |
|---------|-------------------------|
| half (2 bytes) | 16个元素 |
| float (4 bytes) | 8个元素 |
| int32_t (4 bytes) | 8个元素 |

### 5.2 Repeat（迭代）

矢量计算指令执行一次，读取8个DataBlock数据进行计算，称为一个**迭代（Repeat）**。

```
一个Repeat = 8个DataBlock = 8 × 32字节 = 256字节

对于float类型：一个Repeat处理 256/4 = 64个float元素
对于half类型：一个Repeat处理 256/2 = 128个half元素
```

### 5.3 RepeatTimes（迭代次数）

**RepeatTimes**类型为`uint8_t`，**最大值为255**，超过会导致溢出错误。数据量大时需分批处理。

### 5.4 Mask（掩码）

Mask控制每次Repeat内参与计算的元素数量，支持两种模式：

| 模式 | 说明 | 使用场景 |
|------|------|---------|
| **连续模式** | 前面连续的N个元素参与计算 | 处理尾部不足一个Repeat的数据 |
| **逐比特模式** | 每个bit位控制对应元素是否参与 | 需要精确控制哪些元素参与计算 |

### 5.5 概念关系总览

```
┌────────────────────────────────────────────────────────────┐
│                    LocalTensor (UB内存)                      │
│  ┌DB0┐┌DB1┐┌DB2┐┌DB3┐┌DB4┐┌DB5┐┌DB6┐┌DB7┐ ← Repeat 0     │
│  └───┘└───┘└───┘└───┘└───┘└───┘└───┘└───┘                │
│                    ← RepeatStride (DB间隔)                   │
│  ┌DB8┐┌DB9┐┌DB10┐...┌DB15┐             ← Repeat 1         │
│  └───┘└───┘└────┘...└─────┘                               │
│  DataBlockStride: Repeat内相邻DB的间隔                       │
│  Mask: 每个Repeat内参与计算的元素掩码                         │
└────────────────────────────────────────────────────────────┘
```

---

## 6. 常见指令实战演练

### 6.1 算术运算指令实战

除了Add，Membase编程还支持Sub、Mul、Div等算术运算指令。下面我们实现一个**Mul算子**来体验逐元素乘法的编程模式。

In [ ]:
%%writefile Sources/02.06/mul_custom.asc

#include <cstdint>
#include <iostream>
#include <vector>
#include <algorithm>
#include <iterator>
#include "acl/acl.h"
#include "kernel_operator.h"

constexpr uint32_t BUFFER_NUM = 2;

struct MulCustomTilingData
{
    uint32_t totalLength;
    uint32_t tileNum;
};

class KernelMul {
public:
    __aicore__ inline KernelMul(){}
    __aicore__ inline void Init(GM_ADDR x, GM_ADDR y, GM_ADDR z, uint32_t totalLength, uint32_t tileNum);
    __aicore__ inline void Process();

private:
    __aicore__ inline void CopyIn(int32_t progress);
    __aicore__ inline void Compute(int32_t progress);
    __aicore__ inline void CopyOut(int32_t progress);

private:
    AscendC::TPipe pipe;
    AscendC::TQue<AscendC::TPosition::VECIN, BUFFER_NUM> inQueueX, inQueueY;
    AscendC::TQue<AscendC::TPosition::VECOUT, BUFFER_NUM> outQueueZ;
    AscendC::GlobalTensor<float> xGm;
    AscendC::GlobalTensor<float> yGm;
    AscendC::GlobalTensor<float> zGm;
    uint32_t blockLength;
    uint32_t tileNum;
    uint32_t tileLength;
};

__aicore__ inline void KernelMul::Init(GM_ADDR x, GM_ADDR y, GM_ADDR z, uint32_t totalLength, uint32_t tileNum)
{
    this->blockLength = totalLength / AscendC::GetBlockNum();
    this->tileNum = tileNum;
    this->tileLength = this->blockLength / tileNum / BUFFER_NUM;
    xGm.SetGlobalBuffer((__gm__ float *)x + this->blockLength * AscendC::GetBlockIdx(), this->blockLength);
    yGm.SetGlobalBuffer((__gm__ float *)y + this->blockLength * AscendC::GetBlockIdx(), this->blockLength);
    zGm.SetGlobalBuffer((__gm__ float *)z + this->blockLength * AscendC::GetBlockIdx(), this->blockLength);
    pipe.InitBuffer(inQueueX, BUFFER_NUM, this->tileLength * sizeof(float));
    pipe.InitBuffer(inQueueY, BUFFER_NUM, this->tileLength * sizeof(float));
    pipe.InitBuffer(outQueueZ, BUFFER_NUM, this->tileLength * sizeof(float));
}

__aicore__ inline void KernelMul::Process()
{
    int32_t loopCount = this->tileNum * BUFFER_NUM;
    for (int32_t i = 0; i < loopCount; i++) {
        CopyIn(i);
        Compute(i);
        CopyOut(i);
    }
}

__aicore__ inline void KernelMul::CopyIn(int32_t progress)
{
    AscendC::LocalTensor<float> xLocal = inQueueX.AllocTensor<float>();
    AscendC::LocalTensor<float> yLocal = inQueueY.AllocTensor<float>();
    AscendC::DataCopy(xLocal, xGm[progress * this->tileLength], this->tileLength);
    AscendC::DataCopy(yLocal, yGm[progress * this->tileLength], this->tileLength);
    inQueueX.EnQue(xLocal);
    inQueueY.EnQue(yLocal);
}

__aicore__ inline void KernelMul::Compute(int32_t progress)
{
    AscendC::LocalTensor<float> xLocal = inQueueX.DeQue<float>();
    AscendC::LocalTensor<float> yLocal = inQueueY.DeQue<float>();
    AscendC::LocalTensor<float> zLocal = outQueueZ.AllocTensor<float>();
    // 关键差异：使用Mul代替Add，实现逐元素乘法
    AscendC::Mul(zLocal, xLocal, yLocal, this->tileLength);
    outQueueZ.EnQue<float>(zLocal);
    inQueueX.FreeTensor(xLocal);
    inQueueY.FreeTensor(yLocal);
}

__aicore__ inline void KernelMul::CopyOut(int32_t progress)
{
    AscendC::LocalTensor<float> zLocal = outQueueZ.DeQue<float>();
    AscendC::DataCopy(zGm[progress * this->tileLength], zLocal, this->tileLength);
    outQueueZ.FreeTensor(zLocal);
}

__global__ __aicore__ void mul_custom(GM_ADDR x, GM_ADDR y, GM_ADDR z, MulCustomTilingData tiling) {
    KERNEL_TASK_TYPE_DEFAULT(KERNEL_TYPE_AIV_ONLY);
    KernelMul op;
    op.Init(x, y, z, tiling.totalLength, tiling.tileNum);
    op.Process();
}

std::vector<float> kernel_mul(std::vector<float> &x, std::vector<float> &y)
{
    constexpr uint32_t blockDim = 8;
    uint32_t totalLength = x.size();
    size_t totalByteSize = totalLength * sizeof(float);
    int32_t deviceId = 0;
    aclrtStream stream = nullptr;
    MulCustomTilingData tiling = {totalLength, 8};
    uint8_t *xHost = reinterpret_cast<uint8_t *>(x.data());
    uint8_t *yHost = reinterpret_cast<uint8_t *>(y.data());
    uint8_t *zHost = nullptr;
    uint8_t *xDevice = nullptr;
    uint8_t *yDevice = nullptr;
    uint8_t *zDevice = nullptr;

    aclInit(nullptr);
    aclrtSetDevice(deviceId);
    aclrtCreateStream(&stream);
    aclrtMallocHost((void **)(&zHost), totalByteSize);
    aclrtMalloc((void **)&xDevice, totalByteSize, ACL_MEM_MALLOC_HUGE_FIRST);
    aclrtMalloc((void **)&yDevice, totalByteSize, ACL_MEM_MALLOC_HUGE_FIRST);
    aclrtMalloc((void **)&zDevice, totalByteSize, ACL_MEM_MALLOC_HUGE_FIRST);

    aclrtMemcpy(xDevice, totalByteSize, xHost, totalByteSize, ACL_MEMCPY_HOST_TO_DEVICE);
    aclrtMemcpy(yDevice, totalByteSize, yHost, totalByteSize, ACL_MEMCPY_HOST_TO_DEVICE);

    mul_custom<<<blockDim, nullptr, stream>>>(xDevice, yDevice, zDevice, tiling);
    aclrtSynchronizeStream(stream);

    aclrtMemcpy(zHost, totalByteSize, zDevice, totalByteSize, ACL_MEMCPY_DEVICE_TO_HOST);
    std::vector<float> z((float *)zHost, (float *)(zHost + totalByteSize));

    aclrtFree(xDevice);
    aclrtFree(yDevice);
    aclrtFree(zDevice);
    aclrtFreeHost(zHost);
    aclrtDestroyStream(stream);
    aclrtResetDevice(deviceId);
    aclFinalize();
    return z;
}

int32_t main(int32_t argc, char *argv[])
{
    constexpr uint32_t totalLength = 8 * 2048;
    constexpr float valueX = 1.5f;
    constexpr float valueY = 2.0f;
    std::vector<float> x(totalLength, valueX);
    std::vector<float> y(totalLength, valueY);

    std::vector<float> output = kernel_mul(x, y);

    std::vector<float> golden(totalLength, valueX * valueY);
    constexpr size_t maxPrintSize = 20;
    std::cout << "Mul Output: ";
    std::copy(output.begin(), output.begin() + std::min(output.size(), maxPrintSize),
        std::ostream_iterator<float>(std::cout, " "));
    if (output.size() > maxPrintSize) std::cout << "...";
    std::cout << std::endl;
    std::cout << "Mul Golden: ";
    std::copy(golden.begin(), golden.begin() + std::min(golden.size(), maxPrintSize),
        std::ostream_iterator<float>(std::cout, " "));
    if (golden.size() > maxPrintSize) std::cout << "...";
    std::cout << std::endl;
    if (std::equal(golden.begin(), golden.end(), output.begin())) {
        std::cout << "[Success] Mul verification passed." << std::endl;
        return 0;
    } else {
        std::cout << "[Failed] Mul verification failed!" << std::endl;
        return 1;
    }
}

In [ ]:
%%writefile Sources/02.06/CMakeLists_mul.txt

cmake_minimum_required(VERSION 3.16)
find_package(ASC REQUIRED)
project(kernel_samples LANGUAGES ASC CXX)

add_executable(mul_demo
    mul_custom.asc
)
target_compile_options(mul_demo PRIVATE
    $<$<COMPILE_LANGUAGE:ASC>:--npu-arch=dav-2201>
)

In [ ]:
!cd Sources/02.06 && mkdir -p build_mul
!cp Sources/02.06/CMakeLists_mul.txt Sources/02.06/build_mul/CMakeLists.txt
!export ASC_DIR=$ASCEND_HOME_PATH/aarch64-linux/tikcpp/ascendc_kernel_cmake/ && cd Sources/02.06/build_mul/ && cmake .. && make

In [ ]:
!cd Sources/02.06/build_mul/ && ./mul_demo

### 6.2 其他常见指令速查

通过上面的Add和Mul实战，你已经掌握了Membase编程的基本模式。其他指令遵循相同的模式，只是Compute阶段调用的API不同：

| 类别 | 常见指令 | 功能 | Compute阶段代码示例 |
|------|---------|------|-------------------|
| **算术运算** | Add | `z[i] = x[i] + y[i]` | `AscendC::Add(z, x, y, count);` |
| **算术运算** | Sub | `z[i] = x[i] - y[i]` | `AscendC::Sub(z, x, y, count);` |
| **算术运算** | Mul | `z[i] = x[i] * y[i]` | `AscendC::Mul(z, x, y, count);` |
| **算术运算** | Div | `z[i] = x[i] / y[i]` | `AscendC::Div(z, x, y, count);` |
| **标量运算** | Adds | `z[i] = x[i] + scalar` | `AscendC::Adds(z, x, 1.5f, count);` |
| **标量运算** | Muls | `z[i] = x[i] * scalar` | `AscendC::Muls(z, x, 0.5f, count);` |
| **数学运算** | Exp | `z[i] = e^(x[i])` | `AscendC::Exp(z, x, count);` |
| **数学运算** | Log | `z[i] = log(x[i])` | `AscendC::Log(z, x, count);` |
| **数学运算** | Sqrt | `z[i] = sqrt(x[i])` | `AscendC::Sqrt(z, x, count);` |
| **数学运算** | Abs  | `z[i] = abs(x[i])` | `AscendC::Abs(z, x, count);` |
| **类型转换** | Cast | 类型间转换 | `AscendC::Cast(z, x, RoundMode::CAST_NONE, count);` |
| **归约运算** | ReduceSum | `z = sum(x[0..N])` | `AscendC::ReduceSum(z, x, work, count);` |
| **归约运算** | ReduceMax | `z = max(x[0..N])` | `AscendC::ReduceMax(z, x, work, count);` |
| **比较运算** | Max | `z[i] = max(x[i], y[i])` | `AscendC::Max(z, x, y, count);` |

**重要提示**：
- Reduce类API需要一个workLocal临时缓冲区，用于中间计算
- Cast的roundMode常用值：CAST_NONE（不额外舍入）、CAST_RINT（四舍六入五成双）
- FP16精度保护：先Cast到FP32再计算，最后Cast回FP16

---

## 7. Membase编程最佳实践

| 最佳实践 | 说明 |
|---------|------|
| 优先使用DataCopyPad | 不确定对齐时始终用DataCopyPad，避免边界bug |
| FP16精度保护 | 精度敏感计算先Cast到FP32，完成后Cast回FP16 |
| RepeatTimes不超过255 | uint8_t最大255，大数据量需分批处理 |
| 合理使用Double Buffer | `InitBuffer(que, 2, size)` 开启流水并行，但最多4个TQue |
| TBuf vs TQue | MTE搬运用TQue，纯计算用TBuf |
| 禁止高阶封装API | 用基础矢量API，禁止Softmax/LayerNorm等算子级封装 |

---

## 8. Membase vs Regbase

| 维度 | Membase | Regbase |
|------|---------|---------|
| 数据载体 | LocalTensor（UB内存） | RegTensor（寄存器） |
| 数据搬运 | API自动处理 | 手动Load/Store |
| 计算粒度 | 完整LocalTensor | 每次处理VL长度数据 |
| 开发难度 | 低 | 高 |
| 灵活性 | 中 | 极高 |
| 性能上限 | 足够应对大多数场景 | 极致优化场景 |

**切换时机**：日常开发用Membase；需要极致性能或特殊计算模式时考虑Regbase。关于Regbase编程范式的详细介绍，可参考[从一个向量加法出发，深入理解Regbase编程范式](../../blogs/operator/regbase_vec_add/从一个向量加法出发，深入理解Regbase编程范式.md)。

---

## 9. 本节小结

本节系统介绍了SIMD Membase编程的基础知识与常见指令：

1. **Membase定义**：基于内存的编程范式，数据载体为LocalTensor
2. **内存架构**：GM → UB → Vector计算单元的数据流
3. **编程范式**：TPipe + TQue流水线编程，CopyIn → Compute → CopyOut
4. **核心概念**：DataBlock（32字节）、Repeat（8个DataBlock）、Mask、RepeatTimes（≤255）
5. **常见指令**：Add/Sub/Mul/Div、Adds/Muls、Exp/Log/Sqrt/Abs、Cast、ReduceSum、Max/Min
6. **最佳实践**：优先DataCopyPad、FP16精度保护、RepeatTimes限制

---

**参考资料：**
- CANN商用版 9.0.0 - 术语表
- CANN商用版 9.0.0 - Reg矢量计算编程
- CANN商用版 9.0.0 - 基于TPipe和TQue编程

## 课后练习
本节系统介绍了SIMD Membase编程的基础知识与常见指令，请根据学习内容完成以下题目进行自测。

1. （判断题）Membase编程范式中，所有操作均基于内存（Local Memory/Unified Buffer）进行，API自动处理数据搬运过程，开发者无需关心底层寄存器的操作细节。

2. （判断题）在Membase编程范式中，VECIN、VECCALC、VECOUT是物理上不同的存储区域，各自对应不同的硬件存储单元。

3. （单选题）在Membase编程中，一个DataBlock的大小是多少字节？  
    A. 16字节  
    B. 32字节  
    C. 64字节  
    D. 256字节  

4. （单选题）Membase编程中，典型的数据流动路径是？  
    A. GM → Vector计算单元 → UB → GM  
    B. GM → UB(VECIN) → Vector计算 → UB(VECOUT) → GM  
    C. UB → GM → Vector计算单元 → UB  
    D. GM → Vector计算单元 → GM  

5. （单选题）对于float类型，一个Repeat处理多少个元素？  
    A. 8个  
    B. 16个  
    C. 64个  
    D. 128个  

6. （多选题）Membase编程的流水线范式包括以下哪些任务？（可多选）  
    A. CopyIn  
    B. Compute  
    C. CopyOut  
    D. CopyBack  

7. （多选题）以下哪些属于Membase编程的最佳实践？（可多选）  
    A. 优先使用DataCopyPad  
    B. RepeatTimes不超过255  
    C. FP16精度敏感计算先Cast到FP32  
    D. 使用高阶封装API如Softmax、LayerNorm  

**执行以下代码获取答案。**

In [ ]:
!cat ./answer/01.02_answer.txt